# E11 — Physical adjacency with canonical activity and degree controls

This notebook implements the next experiment after E9 and E10. It asks whether
correctly aligned physical message-passing edges add predictive value when
the canonical engineering degree information is supplied consistently.

| Condition | Message edges | Degree covariate | Fit records |
|---|---|---|---:|
| Real | Remaining physical fixation edges | Canonical physical degree | 30 |
| ActiveShuffle01–05 | Same typed edges relabeled within the current active set | Same canonical physical degree | 150 |
| NoEdge | None | Same canonical physical degree | 30 |

**Full study: 210 records.** If all 30 saved E9 DynamicReal GNN anchors pass
verification, only **180 new fits** are trained. Otherwise all 210 are trained.
The existing E6 zero-degree no-edge result is not reused. Model architecture,
semantic labels, operation references, progress, training settings, and the
original simplified E6 state updater stay fixed. No new activity-mask input
is added to the encoder.

**Colab workflow**
1. Upload this notebook through **File → Upload notebook** and select a GPU runtime.
2. Keep your existing `e9_state_aware_baselines_results.zip` on Drive. It is found
   automatically in MyDrive, along with the E9 result-folder location. Set
   `E9_RESULTS_PATH` in Step 2 only if automatic discovery is ambiguous.
3. Run the nine numbered steps in order, or use **Runtime → Run all**.
   **Each numbered step contains exactly one code cell.**
4. Step 6 locks the protocol, validates graph controls, runs one-epoch diagnostics,
   and checks Real anchors. **Step 7 runs the actual full study by default.**
5. Step 8 refreshes summaries and exports `e11_canonical_degree_adjacency_results.zip`.
   Share that ZIP after completion. Step 9 explains the planned comparisons.

Every fit is committed separately to Drive. After a disconnection, reconnect to
the same environment, rerun setup/preflight, and run Step 7: validated completed
fits are skipped. An interrupted fit restarts; earlier completed fits remain.
E9 and E10 files are never overwritten. `RUN_FULL_EXPERIMENT=False` in Step 2
explicitly selects preflight-only use; the default is `True`.

**Interpretation:** the shuffle fixes inactive indices and permutes only active
endpoints using two product/realization-specific fixed priority lists. The
induced permutation stays unchanged while the active set is unchanged. It
preserves each typed graph up to relabeling, including parallel fixation records.
Canonical degree covariates stay fixed even when the computational graph assigns
different per-node degrees. The test concerns edge correspondence with fixed
physical node context, not physical causality. These are teacher-forced reference
rankings, not feasible or optimal disassembly plans. Positive within-GNN effects
would not establish superiority over the E9 non-graph baselines.

## Step 1 — Imports and missing dependencies
Install only missing packages. Keep Colab's existing PyTorch installation and
current runtime. CUBLAS settings are assigned before importing PyTorch.

In [ ]:
import sys,importlib.util,subprocess,os
needed={'numpy':'numpy','pandas':'pandas','torch':'torch',
        'matplotlib':'matplotlib','IPython':'ipython'}
missing=[package for module,package in needed.items() if importlib.util.find_spec(module) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import gc,io,json,math,random,time,tarfile,zipfile,hashlib,itertools,shutil,urllib.request
from pathlib import Path
from collections import Counter,defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display
try:IN_COLAB=importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:IN_COLAB=False
print({'python':sys.version.split()[0],'torch':torch.__version__,'numpy':np.__version__})

## Step 2 — Locked settings and durable output
The default runs the full study after preflight. Use a positive
`MAX_NEW_FITS_THIS_CALL` to pause after a chosen number of new fits; rerunning
Step 7 continues the same study. Leave it as `None` for uninterrupted execution.
Changes to scientific settings create a different run directory. No old output
is deleted. If Drive is already mounted, it is reused.

In [ ]:
SEEDS=[17,29,43]
SHUFFLE_IDS=[1,2,3,4,5]
CONDITIONS=['Real']+[f'ActiveShuffle{i:02d}' for i in SHUFFLE_IDS]+['NoEdge']
EPOCHS=30
NEGATIVES_PER_STATE=8
LR=2e-3
WEIGHT_DECAY=1e-4
DROPOUT=0.10
EMBED_DIM=12
HIDDEN_DIM=64
GNN_DIM=32
BASE_SEED=20261003
GRAPH_SHUFFLE_BASE=20261119
DYNAMIC_SHUFFLE_BASE=GRAPH_SHUFFLE_BASE
PERMUTATION_BASE_SEED=20261011
CPU_THREADS=min(4,os.cpu_count() or 1)
torch.set_num_threads(CPU_THREADS)
torch.use_deterministic_algorithms(True)
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_DRIVE=True
SAVE_MODELS=True
REUSE_E9_REAL=True
RUN_FULL_EXPERIMENT=True
MAX_NEW_FITS_THIS_CALL=None
AUTO_DOWNLOAD_DATASET=True
DATA_PATH=''
E9_RESULTS_PATH=''  # Optional exact ZIP or E9 result directory.
LOCAL_CACHE_ROOT=Path('/content/ev_e11_cache' if IN_COLAB else './ev_e11_cache').resolve()
LOCAL_CACHE_ROOT.mkdir(parents=True,exist_ok=True)
EXPECTED_E9_PROTOCOL=json.loads("{\"experiment\": \"E9 state-aware non-graph baselines\", \"feature_version\": \"E9-state-scalars-v1.0\", \"implementation_sha256\": \"881566fa41124575a6d5ed0f99cb08b518548b4746e32e87e34249164c4f4483\", \"source_e6_sha256\": \"90c01dc01b2eccfb9e592de3ba649c7017a52086d7ee23260db992cb47d48d81\", \"dataset_sha256\": {\"battery_catalogue.csv\": \"a1de7eadff569c511f8aeabf1fa7013bc249f4ad450a7f9bf6a09ca1710371b3\", \"parts.csv\": \"3ceaba9d186dcc7342499e3ccf432a07c78b75dbd4cc732c0b43a92aa140c790\", \"fixations.csv\": \"5bc454a59509275b199828fc7376d28507046bffd28c3f4ec23f076d8bd52339\", \"operations.csv\": \"8cfad668c5b87765dc4de044d72c8b8ce439d67740506a422e4597dc4fc4ba05\"}, \"seeds\": [17, 29, 43], \"epochs\": 30, \"negative_samples\": 8, \"learning_rate\": 0.002, \"weight_decay\": 0.0001, \"dropout\": 0.1, \"xgb_trees\": 250, \"conditions\": [\"StaticFeatureMLP\", \"StateAwareMLP\", \"StaticFeatureXGB\", \"StateAwareXGB\", \"DynamicRealGNN\"], \"folds\": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10], \"torch_version\": \"2.11.0+cu130\", \"xgboost_version\": \"3.4.1\", \"sklearn_version\": \"1.6.1\", \"numpy_version\": \"2.1.3\", \"deterministic_torch\": true, \"tf32_matmul\": false, \"evaluation_batching\": \"Canonical internal batching; shuffled candidate metrics; E6 tie tolerance 1e-8\", \"independent_unit\": \"held-out battery product\", \"state_rule\": \"Exact E6 post-current update; all original fixation classes retained\", \"mlp_capacity\": \"Width chosen by parameter-count gap only; shared by static/state models\", \"xgb_budget\": \"E3 250-tree settings; all nonsingleton training candidates, CPU hist\", \"notes\": \"No ID/order input, no test tuning; historic E6 is descriptive only\"}")
SOURCE_E9_SHA256='023773bb4f03f9de5ebc221578a5aefdd3c2c5697cee048678773b18d82855b2'
IMPLEMENTATION_SHA256='920840198a7ac9ba51d8b4ee8cc22ae64d9306fa0b2e5e7ba1077faac0e9efac'
E11_VERSION='E11-canonical-degree-active-shuffle-v1.0'
METRICS=['hits1','hits3','mrr']
PREFLIGHT_PASSED=False
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive')
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Drive is not mounted. Retry this step or explicitly set USE_DRIVE=False.')
    OUTPUT_ROOT=Path('/content/drive/MyDrive/EV_E11_Canonical_Degree_Adjacency')
else:
    OUTPUT_ROOT=Path('/content/ev_e11' if IN_COLAB else './ev_e11').resolve()
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
assert MAX_NEW_FITS_THIS_CALL is None or (isinstance(MAX_NEW_FITS_THIS_CALL,int) and MAX_NEW_FITS_THIS_CALL>0)
print('Device:',DEVICE,'| output:',OUTPUT_ROOT)
print('Full experiment enabled:',RUN_FULL_EXPERIMENT)
print('Conditions:',CONDITIONS)

## Step 3 — Same public KIT dataset and complete chain initialization
Reuse existing CSVs or the original archive before downloading. All four hashes
must match E9. This single cell loads the data, reconstructs all ten chains,
defines training-only vocabularies, and initializes `FOLD_IDS`.

In [ ]:
CSV_NAMES = ['battery_catalogue.csv', 'parts.csv', 'fixations.csv', 'operations.csv']
DATASET_URL = 'https://radar.kit.edu/radar-backend/archives/emz24pksshndq468/versions/1/content'
search_root = Path('/content') if IN_COLAB else Path.cwd()
def search_local_csvs(root):
    for folder, directories, filenames in os.walk(root):
        directories[:] = [d for d in directories if d not in {'drive', '.git', 'node_modules', '__pycache__'}]
        if set(CSV_NAMES).issubset(filenames):
            return Path(folder)
    return None
def extract_dataset_archive(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:*') as tf:
        resolved = destination.resolve()
        for member in tf.getmembers():
            target = (resolved/member.name).resolve()
            if not target.is_relative_to(resolved) or member.issym() or member.islnk():
                raise ValueError('Unsafe archive entry: '+member.name)
        if sys.version_info >= (3, 12):
            tf.extractall(destination, filter='data')
        else:
            tf.extractall(destination)
    result = search_local_csvs(destination)
    if result is None:
        raise FileNotFoundError('Four dataset CSVs were not found in the archive.')
    return result
if DATA_PATH:
    supplied = Path(DATA_PATH).expanduser()
    if supplied.is_file():
        DATA = extract_dataset_archive(supplied, search_root/'ev_kit_dataset')
    elif supplied.is_dir():
        DATA = search_local_csvs(supplied)
    else:
        raise FileNotFoundError(DATA_PATH)
else:
    DATA = search_local_csvs(search_root)
if DATA is None:
    archives = list(search_root.glob('10.35097-*.tar'))
    if archives:
        DATA = extract_dataset_archive(archives[0], search_root/'ev_kit_dataset')
    elif AUTO_DOWNLOAD_DATASET:
        archive = search_root/'10.35097-emz24pksshndq468.tar'
        if not archive.exists():
            print('Downloading the public KIT dataset ...')
            temp = archive.with_suffix('.tar.part')
            request = urllib.request.Request(DATASET_URL, headers={'User-Agent': 'EV-research-notebook/1.0'})
            with urllib.request.urlopen(request, timeout=90) as response, temp.open('wb') as stream:
                shutil.copyfileobj(response, stream)
            os.replace(temp, archive)
        DATA = extract_dataset_archive(archive, search_root/'ev_kit_dataset')
    else:
        raise FileNotFoundError('Upload the KIT .tar or set DATA_PATH to its extracted CSV directory.')
if DATA is None:
    raise FileNotFoundError('The configured folder does not contain the four KIT CSVs.')
DATASET_HASHES = {name: hashlib.sha256((DATA/name).read_bytes()).hexdigest() for name in CSV_NAMES}
bat = pd.read_csv(DATA/'battery_catalogue.csv')
parts = pd.read_csv(DATA/'parts.csv')
fix = pd.read_csv(DATA/'fixations.csv')
ops = pd.read_csv(DATA/'operations.csv')
# Preserve the column convention used by E6; accept case variations from a CSV export.
rename = {c: {'frompartid': 'fromPartID', 'topartid': 'toPartID'}.get(c.lower(), c) for c in fix.columns}
fix = fix.rename(columns=rename)
actual_counts = (len(bat), len(parts), len(fix), len(ops))
assert actual_counts == (10, 563, 1896, 2281), f'Unexpected dataset version/counts: {actual_counts}'
for frame, required in [(bat, ['id', 'name']), (parts, ['id', 'class', 'battery_id']),
                        (fix, ['id', 'class', 'battery_id', 'fromPartID', 'toPartID']),
                        (ops, ['id', 'skill_id', 'fixation_id', 'part_id', 'next_op_id'])]:
    assert set(required).issubset(frame.columns), set(required)-set(frame.columns)
    assert frame.id.is_unique and frame.id.notna().all()
assert (ops.fixation_id.notna() ^ ops.part_id.notna()).all()
print('Data:', DATA)
print('Counts (products, parts, fixations, operations):', actual_counts)
display(bat[['id', 'name']])

battery_names = bat.set_index('id')['name'].to_dict()
fix_battery = fix.set_index('id')['battery_id'].to_dict()
part_battery = parts.set_index('id')['battery_id'].to_dict()
part_class_raw = parts.set_index('id')['class'].to_dict()
fix_info_raw = fix.set_index('id')[['class', 'fromPartID', 'toPartID']].to_dict('index')

def operation_battery(row):
    if pd.notna(row.fixation_id):
        return int(fix_battery[int(row.fixation_id)])
    return int(part_battery[int(row.part_id)])
ops['battery_id'] = ops.apply(operation_battery, axis=1)
ops_by_id = ops.set_index('id')
chains = {}
for b in sorted(ops['battery_id'].unique()):
    sub = ops[ops['battery_id'] == b]
    ids = set(sub['id'].astype(int))
    next_ids = set(sub['next_op_id'].dropna().astype(int))
    starts = list(ids - next_ids)
    terminals = sub[sub['next_op_id'].isna()]['id'].astype(int).tolist()
    assert len(starts) == 1
    assert len(terminals) == 1
    seq = []
    seen = set()
    cur = int(starts[0])
    while cur in ids and cur not in seen:
        seen.add(cur)
        seq.append(cur)
        nxt = ops_by_id.loc[cur, 'next_op_id']
        if pd.isna(nxt):
            break
        cur = int(nxt)
    assert len(seq) == len(ids), f'Broken chain for battery {b}'
    chains[int(b)] = seq
print({battery_names[b]: len(seq) for b, seq in chains.items()})

def raw_full_signature(op_id):
    r = ops_by_id.loc[int(op_id)]
    skill = str(r.skill_id)
    if pd.notna(r.fixation_id):
        fi = fix_info_raw[int(r.fixation_id)]
        src_cls = str(part_class_raw[int(fi['fromPartID'])])
        dst_cls = str(part_class_raw[int(fi['toPartID'])])
        return f"{skill}|fix:{fi['class']}|{src_cls}->{dst_cls}"
    return f'{skill}|part:{part_class_raw[int(r.part_id)]}'
raw_signatures = {b: [raw_full_signature(op_id) for op_id in seq] for b, seq in chains.items()}

def make_vocab(values):
    vals = sorted({str(x) for x in values if pd.notna(x)})
    return {'<UNK>': 0, **{v: i + 1 for i, v in enumerate(vals)}}

def lookup(vocab, value):
    if pd.isna(value):
        return 0
    return vocab.get(str(value), 0)

def build_fold_vocab(train_batteries):
    return {'part': make_vocab(parts[parts['battery_id'].isin(train_batteries)]['class']), 'fix': make_vocab(fix[fix['battery_id'].isin(train_batteries)]['class']), 'skill': make_vocab(ops[ops['battery_id'].isin(train_batteries)]['skill_id'])}

def build_battery_pack(battery_id, vocab, graph_mode='real', shuffle_id=0):
    assert graph_mode in {'real', 'shuffle', 'noedge'}
    psub = parts[parts['battery_id'] == battery_id].copy()
    fsub = fix[fix['battery_id'] == battery_id].copy()
    part_id_to_local = {int(pid): i for i, pid in enumerate(psub['id'].astype(int))}
    n_nodes = len(psub)
    true_degree = np.zeros(n_nodes, dtype=np.float32)
    edge_src = []
    edge_dst = []
    edge_type = []
    for _, r in fsub.iterrows():
        a = part_id_to_local[int(r.fromPartID)]
        b = part_id_to_local[int(r.toPartID)]
        true_degree[a] += 1
        true_degree[b] += 1
        edge_src += [a, b]
        edge_dst += [b, a]
        et = lookup(vocab['fix'], r['class'])
        edge_type += [et, et]
    true_degree_norm = true_degree / max(1.0, float(true_degree.max()))
    node_part_class = np.array([lookup(vocab['part'], cls) for cls in psub['class']], dtype=np.int64)
    true_edge_index = np.array([edge_src, edge_dst], dtype=np.int64)
    true_edge_type = np.array(edge_type, dtype=np.int64)
    if graph_mode == 'real':
        graph_edge_index = true_edge_index.copy()
        graph_edge_type = true_edge_type.copy()
        graph_degree_norm = true_degree_norm.copy()
    elif graph_mode == 'shuffle':
        rng_graph = np.random.default_rng(GRAPH_SHUFFLE_BASE + 10007 * int(shuffle_id) + int(battery_id))
        permutation = rng_graph.permutation(n_nodes)
        graph_edge_index = permutation[true_edge_index]
        graph_edge_type = true_edge_type.copy()
        shuffled_degree = np.bincount(graph_edge_index.reshape(-1), minlength=n_nodes).astype(np.float32)
        graph_degree_norm = shuffled_degree / max(1.0, float(shuffled_degree.max()))
    else:
        graph_edge_index = np.empty((2, 0), dtype=np.int64)
        graph_edge_type = np.empty((0,), dtype=np.int64)
        graph_degree_norm = np.zeros(n_nodes, dtype=np.float32)
    seq = [int(x) for x in chains[battery_id]]
    categorical = []
    graph_nodes = []
    numeric = []
    signatures = []
    for op_id in seq:
        r = ops_by_id.loc[op_id]
        skill_idx = lookup(vocab['skill'], r.skill_id)
        if pd.notna(r.fixation_id):
            fi = fix_info_raw[int(r.fixation_id)]
            src_pid = int(fi['fromPartID'])
            dst_pid = int(fi['toPartID'])
            src_node = part_id_to_local[src_pid]
            dst_node = part_id_to_local[dst_pid]
            categorical.append([skill_idx, 0, lookup(vocab['fix'], fi['class']), lookup(vocab['part'], part_class_raw[src_pid]), lookup(vocab['part'], part_class_raw[dst_pid]), 0])
            graph_nodes.append([src_node, dst_node, -1])
            numeric.append([true_degree_norm[src_node], true_degree_norm[dst_node], 0.0])
        else:
            pid = int(r.part_id)
            node = part_id_to_local[pid]
            categorical.append([skill_idx, 1, 0, 0, 0, lookup(vocab['part'], part_class_raw[pid])])
            graph_nodes.append([-1, -1, node])
            numeric.append([0.0, 0.0, true_degree_norm[node]])
        signatures.append(raw_full_signature(op_id))
    return {'battery_id': int(battery_id), 'battery_name': battery_names[int(battery_id)], 'node_part_class': node_part_class, 'node_degree': graph_degree_norm.astype(np.float32), 'edge_index': graph_edge_index, 'edge_type': graph_edge_type, 'op_categorical': np.asarray(categorical, dtype=np.int64), 'op_graph_nodes': np.asarray(graph_nodes, dtype=np.int64), 'op_numeric': np.asarray(numeric, dtype=np.float32), 'full_signatures': signatures, 'n_ops': len(seq), 'graph_mode': graph_mode, 'shuffle_id': int(shuffle_id)}

assert DATASET_HASHES==EXPECTED_E9_PROTOCOL['dataset_sha256'],'Dataset version differs from E9.'
FOLD_IDS=sorted(chains)
assert len(FOLD_IDS)==10 and sum(len(seq)-1 for seq in chains.values())==2271
print('Initialized FOLD_IDS:',FOLD_IDS)
print('Expected E11 records:',len(FOLD_IDS)*len(SEEDS)*len(CONDITIONS))

## Step 4 — Original E6 architecture, updater, and E9 evaluation convention
These sources are retained from the delivered E9 notebook. Inactive nodes are
isolated but still have semantic embeddings; no hidden-state activity mask is
introduced. No-edge control inputs are implemented separately in Step 5.

In [ ]:
class BaseOperationRanker(nn.Module):
    """
    Product-invariant MLP operation ranker.

    It never receives battery/product IDs or raw entity IDs.
    """

    def __init__(self, n_skill, n_fix, n_part, emb_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, dropout=DROPOUT):
        super().__init__()
        self.skill_emb = nn.Embedding(n_skill, emb_dim)
        self.kind_emb = nn.Embedding(2, 4)
        self.fix_emb = nn.Embedding(n_fix, emb_dim)
        self.part_emb = nn.Embedding(n_part, emb_dim)
        op_input_dim = emb_dim + 4 + emb_dim + emb_dim + emb_dim + emb_dim + 3
        self.op_mlp = nn.Sequential(nn.Linear(op_input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
        self.score_mlp = nn.Sequential(nn.Linear(hidden_dim * 4 + 1, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))

    def base_operation_encoding(self, pack):
        cats = torch.as_tensor(pack['op_categorical'], dtype=torch.long, device=DEVICE)
        nums = torch.as_tensor(pack['op_numeric'], dtype=torch.float32, device=DEVICE)
        skill = cats[:, 0]
        kind = cats[:, 1]
        fix_class = cats[:, 2]
        src_class = cats[:, 3]
        dst_class = cats[:, 4]
        part_class = cats[:, 5]
        x = torch.cat([self.skill_emb(skill), self.kind_emb(kind), self.fix_emb(fix_class), self.part_emb(src_class), self.part_emb(dst_class), self.part_emb(part_class), nums], dim=-1)
        return self.op_mlp(x)

    def encode_operations(self, pack):
        return self.base_operation_encoding(pack)

    def score_pairs(self, current_repr, candidate_repr, progress):
        x = torch.cat([current_repr, candidate_repr, torch.abs(current_repr - candidate_repr), current_repr * candidate_repr, progress], dim=-1)
        return self.score_mlp(x).squeeze(-1)

class GraphEncoder(nn.Module):
    """
    Two-layer product-invariant message-passing network.
    """

    def __init__(self, n_part, n_fix, part_emb_dim=EMBED_DIM, edge_emb_dim=8, graph_dim=GNN_DIM, dropout=DROPOUT):
        super().__init__()
        self.part_emb = nn.Embedding(n_part, part_emb_dim)
        self.edge_emb = nn.Embedding(n_fix, edge_emb_dim)
        self.input_proj = nn.Linear(part_emb_dim + 1, graph_dim)
        self.message_layers = nn.ModuleList([nn.Linear(graph_dim + edge_emb_dim, graph_dim) for _ in range(2)])
        self.self_layers = nn.ModuleList([nn.Linear(graph_dim, graph_dim) for _ in range(2)])
        self.dropout = dropout

    def forward(self, pack):
        node_class = torch.as_tensor(pack['node_part_class'], dtype=torch.long, device=DEVICE)
        degree = torch.as_tensor(pack['node_degree'], dtype=torch.float32, device=DEVICE).unsqueeze(-1)
        h = F.relu(self.input_proj(torch.cat([self.part_emb(node_class), degree], dim=-1)))
        edge_index = torch.as_tensor(pack['edge_index'], dtype=torch.long, device=DEVICE)
        edge_type = torch.as_tensor(pack['edge_type'], dtype=torch.long, device=DEVICE)
        src = edge_index[0]
        dst = edge_index[1]
        edge_embedding = self.edge_emb(edge_type)
        for message_layer, self_layer in zip(self.message_layers, self.self_layers):
            msg = message_layer(torch.cat([h[src], edge_embedding], dim=-1))
            agg = torch.zeros((h.shape[0], h.shape[1]), dtype=h.dtype, device=DEVICE)
            agg.index_add_(0, dst, msg)
            counts = torch.zeros((h.shape[0], 1), dtype=h.dtype, device=DEVICE)
            counts.index_add_(0, dst, torch.ones((dst.numel(), 1), dtype=h.dtype, device=DEVICE))
            agg = agg / counts.clamp_min(1.0)
            h = F.relu(self_layer(h) + agg)
            h = F.dropout(h, p=self.dropout, training=self.training)
        return h

class GNNOperationRanker(BaseOperationRanker):

    def __init__(self, n_skill, n_fix, n_part, emb_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, graph_dim=GNN_DIM, dropout=DROPOUT):
        super().__init__(n_skill=n_skill, n_fix=n_fix, n_part=n_part, emb_dim=emb_dim, hidden_dim=hidden_dim, dropout=dropout)
        self.graph_encoder = GraphEncoder(n_part=n_part, n_fix=n_fix, part_emb_dim=emb_dim, graph_dim=graph_dim, dropout=dropout)
        self.structural_projection = nn.Sequential(nn.Linear(graph_dim * 3, hidden_dim), nn.ReLU())
        self.combine = nn.Sequential(nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU())

    def encode_operations(self, pack):
        base = self.base_operation_encoding(pack)
        node_h = self.graph_encoder(pack)
        op_nodes = torch.as_tensor(pack['op_graph_nodes'], dtype=torch.long, device=DEVICE)
        n_ops = op_nodes.shape[0]
        a = torch.zeros((n_ops, node_h.shape[1]), device=DEVICE)
        b = torch.zeros_like(a)
        fixation_mask = op_nodes[:, 0] >= 0
        if fixation_mask.any():
            a[fixation_mask] = node_h[op_nodes[fixation_mask, 0]]
            b[fixation_mask] = node_h[op_nodes[fixation_mask, 1]]
        part_mask = op_nodes[:, 2] >= 0
        if part_mask.any():
            a[part_mask] = node_h[op_nodes[part_mask, 2]]
        structural = self.structural_projection(torch.cat([a, b, torch.abs(a - b)], dim=-1))
        return self.combine(torch.cat([base, structural], dim=-1))

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_training_pairs(n_ops, k, rng):
    current, positive, negative, progress = ([], [], [], [])
    for t in range(n_ops - 2):
        pool = np.arange(t + 2, n_ops, dtype=np.int64)
        neg = rng.choice(pool, size=k, replace=len(pool) < k)
        current.append(t)
        positive.append(t + 1)
        negative.append(neg)
        progress.append(t / max(1, n_ops - 1))
    return (np.asarray(current, dtype=np.int64), np.asarray(positive, dtype=np.int64), np.stack(negative), np.asarray(progress, dtype=np.float32))

def expected_tie_metrics(scores, target_position=0, tol=1e-08):
    """
    Scores correspond to candidate operations.

    Returns the expected Hits@1, Hits@3, and reciprocal rank
    under random ordering within exact/near score ties.
    """
    scores = np.asarray(scores)
    target_score = scores[target_position]
    greater = int(np.sum(scores > target_score + tol))
    equal = int(np.sum(np.abs(scores - target_score) <= tol))
    equal = max(equal, 1)
    hits1 = max(0, min(equal, 1 - greater)) / equal if greater < 1 else 0.0
    hits3 = max(0, min(equal, 3 - greater)) / equal if greater < 3 else 0.0
    ranks = np.arange(greater + 1, greater + equal + 1, dtype=float)
    mrr = float(np.mean(1.0 / ranks))
    return (hits1, hits3, mrr)

def count_trainable_parameters(model):
    return sum((p.numel() for p in model.parameters() if p.requires_grad))

def training_signature_set(train_batteries):
    s = set()
    for b in train_batteries:
        s.update(raw_signatures[b])
    return s

from collections import defaultdict
DYNAMIC_SHUFFLE_BASE = GRAPH_SHUFFLE_BASE

def _battery_structure_records(battery_id, vocab):
    psub = parts[parts['battery_id'] == battery_id].copy()
    fsub = fix[fix['battery_id'] == battery_id].copy()
    part_ids = psub['id'].astype(int).tolist()
    part_id_to_local = {pid: i for i, pid in enumerate(part_ids)}
    records = []
    for _, r in fsub.iterrows():
        records.append({'fixation_id': int(r['id']), 'u': part_id_to_local[int(r.fromPartID)], 'v': part_id_to_local[int(r.toPartID)], 'edge_type': lookup(vocab['fix'], r['class'])})
    return (psub, part_id_to_local, records)

def _dynamic_permutation(battery_id, n_nodes, shuffle_id):
    rng = np.random.default_rng(DYNAMIC_SHUFFLE_BASE + 10007 * int(shuffle_id) + int(battery_id))
    return rng.permutation(n_nodes)

def _state_edges_and_degree(n_nodes, records, removed_fixations, active_nodes, initial_max_degree):
    edge_src = []
    edge_dst = []
    edge_type = []
    degree = np.zeros(n_nodes, dtype=np.float32)
    for rec in records:
        if rec['fixation_id'] in removed_fixations:
            continue
        u = int(rec['u'])
        v = int(rec['v'])
        if not active_nodes[u] or not active_nodes[v]:
            continue
        degree[u] += 1.0
        degree[v] += 1.0
        edge_src += [u, v]
        edge_dst += [v, u]
        edge_type += [int(rec['edge_type']), int(rec['edge_type'])]
    if len(edge_src):
        edge_index = np.asarray([edge_src, edge_dst], dtype=np.int64)
        edge_type = np.asarray(edge_type, dtype=np.int64)
    else:
        edge_index = np.empty((2, 0), dtype=np.int64)
        edge_type = np.empty((0,), dtype=np.int64)
    degree_norm = (degree / max(1.0, float(initial_max_degree))).astype(np.float32)
    return (edge_index, edge_type, degree_norm)

def _complete_state_edges_and_degree(active_nodes):
    active = np.flatnonzero(active_nodes).astype(np.int64)
    n_nodes = len(active_nodes)
    if len(active) <= 1:
        return (np.empty((2, 0), dtype=np.int64), np.empty((0,), dtype=np.int64), np.zeros(n_nodes, dtype=np.float32))
    src = np.repeat(active, len(active))
    dst = np.tile(active, len(active))
    keep = src != dst
    src = src[keep]
    dst = dst[keep]
    edge_index = np.asarray([src, dst], dtype=np.int64)
    edge_type = np.zeros(len(src), dtype=np.int64)
    degree_norm = np.zeros(n_nodes, dtype=np.float32)
    degree_norm[active] = (len(active) - 1) / max(1, n_nodes - 1)
    return (edge_index, edge_type, degree_norm)

def _has_remaining_incident_fixation(node, records, removed_fixations, active_nodes):
    """
    Causal check: does this currently active node still participate in any
    modeled fixation that has not yet been executed and whose other endpoint
    is also structurally active?
    """
    for rec in records:
        if rec['fixation_id'] in removed_fixations:
            continue
        u = int(rec['u'])
        v = int(rec['v'])
        if not active_nodes[u] or not active_nodes[v]:
            continue
        if node == u or node == v:
            return True
    return False

def _deactivate_completed_parts(records, removed_fixations, active_nodes, part_completed):
    """
    Deactivate a part only after:
      1) its part operation has already occurred, and
      2) it has no remaining active incident fixation.

    Iterate to a fixed point because deactivating one completed isolated node
    can make another completed node isolated as well.
    """
    changed = True
    while changed:
        changed = False
        pending = np.flatnonzero(part_completed & active_nodes)
        for node in pending:
            node = int(node)
            if not _has_remaining_incident_fixation(node=node, records=records, removed_fixations=removed_fixations, active_nodes=active_nodes):
                active_nodes[node] = False
                changed = True
    return active_nodes

def build_e6_state_pack(battery_id, vocab, condition, shuffle_id=0):
    """
    Conditions:
      static_real
      dynamic_real
      dynamic_noedge
      dynamic_complete
      dynamic_shuffle

    Returns the invariant operation pack plus one block-diagonal graph
    containing all post-current ranking states t=0,...,n_ops-2.
    """
    assert condition in {'static_real', 'dynamic_real', 'dynamic_noedge', 'dynamic_complete', 'dynamic_shuffle'}
    base_pack = build_battery_pack(battery_id, vocab, graph_mode='real', shuffle_id=0)
    psub, part_id_to_local, true_records = _battery_structure_records(battery_id, vocab)
    n_nodes = len(psub)
    n_states = base_pack['n_ops'] - 1
    initial_degree = np.zeros(n_nodes, dtype=np.float32)
    for rec in true_records:
        initial_degree[rec['u']] += 1.0
        initial_degree[rec['v']] += 1.0
    initial_max_degree = max(1.0, float(initial_degree.max()))
    if condition == 'dynamic_shuffle':
        permutation = _dynamic_permutation(battery_id, n_nodes, shuffle_id)
        records = [{'fixation_id': rec['fixation_id'], 'u': int(permutation[rec['u']]), 'v': int(permutation[rec['v']]), 'edge_type': rec['edge_type']} for rec in true_records]
        event_node_map = permutation
    else:
        permutation = np.arange(n_nodes, dtype=np.int64)
        records = [dict(rec) for rec in true_records]
        event_node_map = permutation
    static_edge_index, static_edge_type, static_degree = _state_edges_and_degree(n_nodes=n_nodes, records=records, removed_fixations=set(), active_nodes=np.ones(n_nodes, dtype=bool), initial_max_degree=initial_max_degree)
    removed_fixations = set()
    active_nodes = np.ones(n_nodes, dtype=bool)
    part_completed = np.zeros(n_nodes, dtype=bool)
    state_edge_indices = []
    state_edge_types = []
    state_degrees = []
    state_active_fraction = []
    state_edge_fraction = []
    initial_undirected_edges = len(true_records)
    seq = [int(x) for x in chains[battery_id]]
    for t in range(n_states):
        op_id = seq[t]
        r = ops_by_id.loc[op_id]
        if condition != 'static_real':
            if pd.notna(r.fixation_id):
                removed_fixations.add(int(r.fixation_id))
            else:
                semantic_node = part_id_to_local[int(r.part_id)]
                structural_node = int(event_node_map[semantic_node])
                part_completed[structural_node] = True
            active_nodes = _deactivate_completed_parts(records=records, removed_fixations=removed_fixations, active_nodes=active_nodes, part_completed=part_completed)
        if condition == 'static_real':
            edge_index = static_edge_index
            edge_type = static_edge_type
            degree_norm = static_degree
            active_fraction = 1.0
            edge_fraction = 1.0
        elif condition == 'dynamic_noedge':
            edge_index = np.empty((2, 0), dtype=np.int64)
            edge_type = np.empty((0,), dtype=np.int64)
            degree_norm = np.zeros(n_nodes, dtype=np.float32)
            active_fraction = float(active_nodes.mean())
            edge_fraction = 0.0
        elif condition == 'dynamic_complete':
            edge_index, edge_type, degree_norm = _complete_state_edges_and_degree(active_nodes)
            active_fraction = float(active_nodes.mean())
            max_directed = n_nodes * max(0, n_nodes - 1)
            edge_fraction = edge_index.shape[1] / max(1, max_directed)
        else:
            edge_index, edge_type, degree_norm = _state_edges_and_degree(n_nodes=n_nodes, records=records, removed_fixations=removed_fixations, active_nodes=active_nodes, initial_max_degree=initial_max_degree)
            active_fraction = float(active_nodes.mean())
            edge_fraction = edge_index.shape[1] / 2.0 / max(1, initial_undirected_edges)
        state_edge_indices.append(edge_index)
        state_edge_types.append(edge_type)
        state_degrees.append(degree_norm)
        state_active_fraction.append(active_fraction)
        state_edge_fraction.append(edge_fraction)
    block_src = []
    block_dst = []
    block_type = []
    for state_idx, (edge_index, edge_type) in enumerate(zip(state_edge_indices, state_edge_types)):
        offset = state_idx * n_nodes
        if edge_index.shape[1]:
            block_src.append(edge_index[0] + offset)
            block_dst.append(edge_index[1] + offset)
            block_type.append(edge_type)
    if block_src:
        block_edge_index = np.asarray([np.concatenate(block_src), np.concatenate(block_dst)], dtype=np.int64)
        block_edge_type = np.concatenate(block_type).astype(np.int64)
    else:
        block_edge_index = np.empty((2, 0), dtype=np.int64)
        block_edge_type = np.empty((0,), dtype=np.int64)
    block_pack = {'node_part_class': np.tile(base_pack['node_part_class'], n_states).astype(np.int64), 'node_degree': np.concatenate(state_degrees).astype(np.float32), 'edge_index': block_edge_index, 'edge_type': block_edge_type}
    return {'base_pack': base_pack, 'state_graph_pack': block_pack, 'n_nodes': int(n_nodes), 'n_states': int(n_states), 'state_active_fraction': np.asarray(state_active_fraction, dtype=np.float32), 'state_edge_fraction': np.asarray(state_edge_fraction, dtype=np.float32), 'condition': condition, 'shuffle_id': int(shuffle_id), 'permutation': permutation.astype(np.int64)}

def build_e6_model(vocab):
    return GNNOperationRanker(n_skill=len(vocab['skill']), n_fix=len(vocab['fix']), n_part=len(vocab['part'])).to(DEVICE)

def compose_state_operation_repr(model, base_repr, node_h, base_pack, n_nodes, state_indices, op_indices):
    state_indices = torch.as_tensor(state_indices, dtype=torch.long, device=DEVICE)
    op_indices = torch.as_tensor(op_indices, dtype=torch.long, device=DEVICE)
    op_nodes_all = torch.as_tensor(base_pack['op_graph_nodes'], dtype=torch.long, device=DEVICE)
    op_nodes = op_nodes_all[op_indices]
    offsets = state_indices * int(n_nodes)
    n_items = len(op_indices)
    a = torch.zeros((n_items, node_h.shape[1]), dtype=node_h.dtype, device=DEVICE)
    b = torch.zeros_like(a)
    fixation_mask = op_nodes[:, 0] >= 0
    if fixation_mask.any():
        a[fixation_mask] = node_h[offsets[fixation_mask] + op_nodes[fixation_mask, 0]]
        b[fixation_mask] = node_h[offsets[fixation_mask] + op_nodes[fixation_mask, 1]]
    part_mask = op_nodes[:, 2] >= 0
    if part_mask.any():
        a[part_mask] = node_h[offsets[part_mask] + op_nodes[part_mask, 2]]
    structural = model.structural_projection(torch.cat([a, b, torch.abs(a - b)], dim=-1))
    return model.combine(torch.cat([base_repr[op_indices], structural], dim=-1))

def train_one_e6_condition(train_batteries, test_battery, seed, condition, shuffle_id=0):
    set_seed(seed)
    vocab = build_fold_vocab(train_batteries)
    packs = {b: build_e6_state_pack(b, vocab, condition=condition, shuffle_id=shuffle_id) for b in train_batteries + [test_battery]}
    model = build_e6_model(vocab)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    rng = np.random.default_rng(seed)
    for epoch in range(EPOCHS):
        order = list(train_batteries)
        rng.shuffle(order)
        model.train()
        for b in order:
            item = packs[b]
            base_pack = item['base_pack']
            n_ops = base_pack['n_ops']
            cur, pos, neg, prog = make_training_pairs(n_ops, NEGATIVES_PER_STATE, rng)
            state_idx = cur.copy()
            base_repr = model.base_operation_encoding(base_pack)
            node_h = model.graph_encoder(item['state_graph_pack'])
            cur_repr = compose_state_operation_repr(model=model, base_repr=base_repr, node_h=node_h, base_pack=base_pack, n_nodes=item['n_nodes'], state_indices=state_idx, op_indices=cur)
            pos_repr = compose_state_operation_repr(model=model, base_repr=base_repr, node_h=node_h, base_pack=base_pack, n_nodes=item['n_nodes'], state_indices=state_idx, op_indices=pos)
            prog_t = torch.as_tensor(prog, dtype=torch.float32, device=DEVICE).unsqueeze(-1)
            pos_score = model.score_pairs(cur_repr, pos_repr, prog_t)
            k = neg.shape[1]
            neg_flat = neg.reshape(-1)
            neg_state = np.repeat(state_idx, k)
            neg_repr = compose_state_operation_repr(model=model, base_repr=base_repr, node_h=node_h, base_pack=base_pack, n_nodes=item['n_nodes'], state_indices=neg_state, op_indices=neg_flat)
            cur_rep = cur_repr[:, None, :].expand(-1, k, -1).reshape(-1, cur_repr.shape[1])
            prog_rep = prog_t[:, None, :].expand(-1, k, -1).reshape(-1, 1)
            neg_score = model.score_pairs(cur_rep, neg_repr, prog_rep).reshape(len(cur), k)
            loss = F.softplus(-(pos_score[:, None] - neg_score)).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return (model, packs[test_battery], vocab)

def evaluate_e6_model(model, item, train_batteries):
    model.eval()
    base_pack = item['base_pack']
    seen_signatures = training_signature_set(train_batteries)
    rows = []
    with torch.no_grad():
        base_repr = model.base_operation_encoding(base_pack)
        node_h = model.graph_encoder(item['state_graph_pack'])
        n_ops = base_pack['n_ops']
        for t in range(n_ops - 1):
            candidate_indices = np.arange(t + 1, n_ops, dtype=np.int64)
            state_current = np.asarray([t], dtype=np.int64)
            current_repr = compose_state_operation_repr(model=model, base_repr=base_repr, node_h=node_h, base_pack=base_pack, n_nodes=item['n_nodes'], state_indices=state_current, op_indices=np.asarray([t], dtype=np.int64))
            cand_states = np.full(len(candidate_indices), t, dtype=np.int64)
            candidate_repr = compose_state_operation_repr(model=model, base_repr=base_repr, node_h=node_h, base_pack=base_pack, n_nodes=item['n_nodes'], state_indices=cand_states, op_indices=candidate_indices)
            current_rep = current_repr.expand(len(candidate_indices), -1)
            progress = torch.full((len(candidate_indices), 1), t / max(1, n_ops - 1), dtype=torch.float32, device=DEVICE)
            scores = model.score_pairs(current_rep, candidate_repr, progress).cpu().numpy()
            h1, h3, mrr = expected_tie_metrics(scores, target_position=0)
            target_signature = base_pack['full_signatures'][t + 1]
            rows.append({'position': int(t), 'hits1': float(h1), 'hits3': float(h3), 'mrr': float(mrr), 'signature_seen_in_training': bool(target_signature in seen_signatures), 'active_node_fraction': float(item['state_active_fraction'][t]), 'edge_fraction': float(item['state_edge_fraction'][t])})
    detail = pd.DataFrame(rows)
    metrics = {'hits1': float(detail['hits1'].mean()), 'hits3': float(detail['hits3'].mean()), 'mrr': float(detail['mrr'].mean()), 'seen_signature_rate': float(detail['signature_seen_in_training'].mean())}
    return (metrics, detail)

def candidate_inventory(n_ops, prefix):
    # Sequence indices are internal lookup keys, never numeric model inputs.
    return np.setdiff1d(np.arange(n_ops), np.asarray(prefix), assume_unique=True)

def evaluate_scalar_or_gnn(model, item, kind, state_aware=None, encoder=None, candidate_seed=None):
    base = item["base_pack"]
    n_ops = base["n_ops"]
    rng = np.random.default_rng(BASE_SEED + 1009 * int(base["battery_id"])
                                if candidate_seed is None else candidate_seed)
    if kind in {"mlp", "gnn"}:
        model.eval()
    rows = []
    with torch.no_grad():
        if kind == "gnn":
            base_repr = model.base_operation_encoding(base)
            node_h = model.graph_encoder(item["state_graph_pack"])
        for t in range(n_ops-1):
            inventory = candidate_inventory(n_ops, range(t+1))
            candidates = rng.permutation(inventory)
            # Fixed internal batching preserves E6's floating-point tie behavior.
            # Lookup order is never a feature. Shuffle scores back before the metric.
            scoring_candidates = inventory
            states = np.full(len(candidates), t, dtype=np.int64)
            if kind == "mlp":
                scores = model.score_selected(item, states, states, scoring_candidates, state_aware).cpu().numpy()
            elif kind == "xgb":
                scores = model.predict(scalar_pair_matrix(item, t, scoring_candidates, encoder, state_aware))
            elif kind == "gnn":
                current = compose_state_operation_repr(model, base_repr, node_h, base,
                            item["n_nodes"], np.array([t]), np.array([t])).expand(len(candidates), -1)
                candidate = compose_state_operation_repr(model, base_repr, node_h, base,
                            item["n_nodes"], states, scoring_candidates)
                progress = torch.full((len(candidates), 1), t/max(1, n_ops-1), device=DEVICE)
                scores = model.score_pairs(current, candidate, progress).cpu().numpy()
            else:
                raise ValueError(kind)
            if not np.isfinite(scores).all():
                raise FloatingPointError("Non-finite evaluation scores")
            scores = scores[np.searchsorted(inventory, candidates)]
            target = int(np.flatnonzero(candidates == t+1)[0])
            h1, h3, mrr = expected_tie_metrics(scores, target_position=target)
            rows.append({"position": t, "candidate_size": len(candidates),
                         "progress": t/max(1, n_ops-1), "hits1": h1, "hits3": h3, "mrr": mrr})
    detail = pd.DataFrame(rows)
    return {m: float(detail[m].mean()) for m in METRICS}, detail

## Step 5 — E11 controls, checkpointing, anchor verification, and analysis
The added logic transforms message edges after the canonical prefix update.
Real reproduces E9; NoEdge retains canonical degrees. Source checkpoints are
reused only if all 30 Real anchors pass. Partial studies cannot produce primary
significance values. The complete run uses two Hits@1 tests in one Holm family.

In [ ]:
def atomic_json(path, payload):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temp, path)

def atomic_csv(frame, path):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    compression = "gzip" if str(path).endswith(".gz") else None
    frame.to_csv(temp, index=False, compression=compression)
    os.replace(temp, path)

def fit_paths(run_dir, battery_id, seed, condition):
    key = f"battery{int(battery_id):02d}_seed{int(seed)}_{condition}"
    return key, run_dir/"fits"/(key+".json"), run_dir/"fits"/(key+"_states.csv.gz")

def exact_sign_flip(values):
    values = np.asarray(values, dtype=float)
    observed = abs(values.mean())
    means = [abs(np.mean(values*np.array(signs))) for signs in itertools.product([-1, 1], repeat=len(values))]
    return float(np.mean(np.asarray(means) >= observed-1e-12))

def holm(pvalues):
    values = np.asarray(pvalues, dtype=float)
    order = np.argsort(values)
    result = np.empty(len(values))
    last = 0.0
    for rank, idx in enumerate(order):
        last = max(last, min(1.0, (len(values)-rank)*values[idx]))
        result[idx] = last
    return result

def canonical_activity_history(battery_id, vocab):
    _, part_map, records = _battery_structure_records(battery_id, vocab)
    active = np.ones(len(part_map), dtype=bool)
    completed = np.zeros(len(part_map), dtype=bool)
    removed = set()
    history = []
    for op_id in chains[battery_id][:-1]:
        row = ops_by_id.loc[int(op_id)]
        if pd.notna(row.fixation_id):
            removed.add(int(row.fixation_id))
        else:
            completed[part_map[int(row.part_id)]] = True
        _deactivate_completed_parts(records, removed, active, completed)
        history.append(active.copy())
    return np.stack(history)


def priority_lists(battery_id, n_nodes, realization, identity=False):
    rng = np.random.default_rng(np.random.SeedSequence(
        [PERMUTATION_BASE_SEED, int(battery_id), int(realization)]))
    a = rng.permutation(n_nodes)
    b = a.copy() if identity else rng.permutation(n_nodes)
    return a, b


def active_permutation(active, first, second):
    mapping = np.arange(len(active), dtype=np.int64)
    mapping[first[active[first]]] = second[active[second]]
    assert np.array_equal(mapping[~active], np.flatnonzero(~active))
    assert np.array_equal(np.sort(mapping[active]), np.flatnonzero(active))
    return mapping


def typed_edge_codes(edges, types, n_nodes, n_types):
    return np.sort((edges[0] * n_nodes + edges[1]) * n_types + types)


def build_e11_state_pack(battery_id, vocab, condition, identity=False, canonical=None):
    """Transform message edges only; all node/operation inputs stay canonical."""
    if condition not in CONDITIONS:
        raise ValueError(condition)
    real = (build_e6_state_pack(battery_id, vocab, 'dynamic_real')
            if canonical is None else canonical)
    active = canonical_activity_history(battery_id, vocab)
    n, states = real['n_nodes'], real['n_states']
    np.testing.assert_allclose(active.mean(axis=1), real['state_active_fraction'], rtol=0, atol=1e-7)
    graph = {k: value.copy() for k, value in real['state_graph_pack'].items()}
    original = graph['edge_index']
    edge_states = original[0] // n
    assert np.array_equal(edge_states, original[1] // n)
    assert np.all(np.diff(edge_states) >= 0)
    boundaries = np.r_[0, np.cumsum(np.bincount(edge_states, minlength=states))]
    n_types = len(vocab['fix'])
    realization = int(condition.removeprefix('ActiveShuffle')) if condition.startswith('ActiveShuffle') else 0
    first, second = priority_lists(battery_id, n, realization, identity)
    mappings = np.tile(np.arange(n, dtype=np.int64), (states, 1))
    diagnostics = []
    previous_active, previous_mapping = None, None
    for t in range(states):
        sl = slice(int(boundaries[t]), int(boundaries[t+1]))
        old_edges = original[:, sl] - t*n
        types = graph['edge_type'][sl]
        if condition.startswith('ActiveShuffle'):
            if previous_active is None or not np.array_equal(active[t], previous_active):
                previous_mapping = active_permutation(active[t], first, second)
                previous_active = active[t].copy()
            mapping = previous_mapping
            mappings[t] = mapping
            new_edges = mapping[old_edges]
            graph['edge_index'][:, sl] = new_edges + t*n
        elif condition == 'NoEdge':
            new_edges = np.empty((2, 0), dtype=np.int64)
        else:
            new_edges = old_edges
        old_degree = np.bincount(old_edges[0], minlength=n)
        new_degree = np.bincount(new_edges[0], minlength=n)
        if condition.startswith('ActiveShuffle'):
            assert np.array_equal(np.sort(old_degree), np.sort(new_degree))
            assert not new_edges.size or active[t, new_edges].all()
            expected = mappings[t][old_edges]
            assert np.array_equal(new_edges, expected)
        actual_types = np.empty(0, dtype=np.int64) if condition == 'NoEdge' else types
        changed = not np.array_equal(typed_edge_codes(old_edges, types, n, n_types),
                                     typed_edge_codes(new_edges, actual_types, n, n_types))
        diagnostics.append({'position': t, 'active_nodes': int(active[t].sum()),
            'canonical_directed_edges': int(old_edges.shape[1]),
            'message_directed_edges': int(new_edges.shape[1]),
            'moved_active_nodes': int(np.sum(mappings[t] != np.arange(n))),
            'edge_record_endpoint_change_fraction': (float(np.any(new_edges != old_edges, axis=0).mean())
                if old_edges.shape[1] and condition != 'NoEdge' else (1.0 if old_edges.shape[1] else 0.0)),
            'typed_adjacency_changed': bool(changed),
            'computational_degree_mismatch_fraction': float(np.mean(new_degree != old_degree)),
            'trivially_nonperturbable': bool(active[t].sum() < 2 or old_edges.shape[1] == 0)})
    if condition == 'NoEdge':
        graph['edge_index'] = np.empty((2, 0), dtype=np.int64)
        graph['edge_type'] = np.empty(0, dtype=np.int64)
    np.testing.assert_array_equal(graph['node_degree'], real['state_graph_pack']['node_degree'])
    np.testing.assert_array_equal(graph['node_part_class'], real['state_graph_pack']['node_part_class'])
    if condition.startswith('ActiveShuffle'):
        np.testing.assert_array_equal(graph['edge_type'], real['state_graph_pack']['edge_type'])
    item = dict(real)
    item.update(state_graph_pack=graph, condition=condition, shuffle_id=realization,
                canonical_active=active, active_permutations=mappings,
                perturbation_diagnostics=pd.DataFrame(diagnostics),
                physical_state_edge_fraction=real['state_edge_fraction'].copy())
    if condition == 'NoEdge':
        item['state_edge_fraction'] = np.zeros(states, dtype=np.float32)
    return item


def assert_same_graph_inputs(a, b):
    for k in ['node_part_class', 'node_degree', 'edge_index', 'edge_type']:
        np.testing.assert_array_equal(a['state_graph_pack'][k], b['state_graph_pack'][k])
    for k in ['op_numeric', 'op_categorical', 'op_graph_nodes']:
        np.testing.assert_array_equal(a['base_pack'][k], b['base_pack'][k])


def train_e11(train_batteries, test_battery, seed, condition, epochs=None):
    # Same model initialization, order, negative sampling, optimizer, and updates as E6.
    set_seed(seed)
    vocab = build_fold_vocab(train_batteries)
    packs = {b: build_e11_state_pack(b, vocab, condition) for b in train_batteries + [test_battery]}
    model = build_e6_model(vocab)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    rng = np.random.default_rng(seed)
    losses = []
    for epoch in range(EPOCHS if epochs is None else epochs):
        order = list(train_batteries)
        rng.shuffle(order)
        model.train()
        epoch_losses = []
        for b in order:
            item = packs[b]
            base = item['base_pack']
            cur, pos, neg, prog = make_training_pairs(base['n_ops'], NEGATIVES_PER_STATE, rng)
            state = cur.copy()
            base_repr = model.base_operation_encoding(base)
            node_h = model.graph_encoder(item['state_graph_pack'])
            current = compose_state_operation_repr(model, base_repr, node_h, base, item['n_nodes'], state, cur)
            positive = compose_state_operation_repr(model, base_repr, node_h, base, item['n_nodes'], state, pos)
            progress = torch.as_tensor(prog, dtype=torch.float32, device=DEVICE).unsqueeze(-1)
            pos_score = model.score_pairs(current, positive, progress)
            k = neg.shape[1]
            negative = compose_state_operation_repr(model, base_repr, node_h, base, item['n_nodes'],
                                                      np.repeat(state, k), neg.reshape(-1))
            repeated = current[:, None, :].expand(-1, k, -1).reshape(-1, current.shape[1])
            prog_repeat = progress[:, None, :].expand(-1, k, -1).reshape(-1, 1)
            neg_score = model.score_pairs(repeated, negative, prog_repeat).reshape(len(cur), k)
            loss = F.softplus(-(pos_score[:, None] - neg_score)).mean()
            if not torch.isfinite(loss):
                raise FloatingPointError(f'Nonfinite loss: product {b}, epoch {epoch+1}, {condition}')
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_losses.append(float(loss.detach().cpu()))
        losses.append({'epoch': epoch+1, 'loss': float(np.mean(epoch_losses))})
    return model, packs[test_battery], vocab, losses


def evaluate_e11(model, item, candidate_seed=None):
    metrics, frame = evaluate_scalar_or_gnn(model, item, 'gnn', candidate_seed=candidate_seed)
    frame = frame.merge(item['perturbation_diagnostics'], on='position', validate='one_to_one')
    return metrics, frame


def compare_raw_scores(model, real, identity):
    """Identity relabeling must preserve the actual graph activations and scores."""
    model.eval()
    assert_same_graph_inputs(real, identity)
    with torch.no_grad():
        left = model.graph_encoder(real['state_graph_pack'])
        right = model.graph_encoder(identity['state_graph_pack'])
        assert torch.equal(left, right)
        base = model.base_operation_encoding(real['base_pack'])
        for t in range(real['n_states']):
            candidates = np.arange(t+1, real['base_pack']['n_ops'])
            state = np.full(len(candidates), t)
            curr = np.array([t])
            progress = torch.full((len(candidates), 1), t/max(1, real['n_states']), device=DEVICE)
            results = []
            for h in [left, right]:
                a = compose_state_operation_repr(model, base, h, real['base_pack'], real['n_nodes'], curr, curr)
                b = compose_state_operation_repr(model, base, h, real['base_pack'], real['n_nodes'], state, candidates)
                results.append(model.score_pairs(a.expand(len(candidates), -1), b, progress))
            assert torch.equal(*results)


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()


def resolve_e9_source():
    if E9_RESULTS_PATH:
        candidate = Path(E9_RESULTS_PATH).expanduser()
        if not candidate.exists():
            raise FileNotFoundError('E9_RESULTS_PATH does not exist: '+str(candidate))
    else:
        locations = [Path('/content'), Path('/content/drive/MyDrive')] if IN_COLAB else [Path.cwd()]
        candidates = [p for root in locations for p in root.glob('e9_state_aware_baselines_results*.zip')]
        if len(candidates) > 1:
            raise RuntimeError('Multiple E9 ZIPs found. Set E9_RESULTS_PATH to the intended file.')
        if candidates:
            candidate = candidates[0]
        else:
            parents = [Path('/content/drive/MyDrive/EV_State_Aware_Baselines'),
                       Path('/content/ev_state_baselines')] if IN_COLAB else [Path('./ev_state_baselines')]
            found = [p for p in parents if p.is_dir()]
            if not found:
                return None
            if len(found) > 1:
                raise RuntimeError('Multiple E9 result roots found. Set E9_RESULTS_PATH explicitly.')
            candidate = found[0]
    if candidate.is_file():
        if not zipfile.is_zipfile(candidate):
            raise ValueError('E9 source must be a result ZIP or result directory.')
        destination = LOCAL_CACHE_ROOT/('e9_'+sha256_file(candidate)[:16])
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(candidate) as z:
            base = destination.resolve()
            for entry in z.infolist():
                target = (base/entry.filename).resolve()
                if not target.is_relative_to(base) or ((entry.external_attr >> 16) & 0o170000) == 0o120000:
                    raise ValueError('Unsafe ZIP entry: '+entry.filename)
            z.extractall(destination)
        candidate = destination
    runs = [candidate] if (candidate/'protocol.json').is_file() else sorted(
        p.parent for p in candidate.rglob('protocol.json') if (p.parent/'e9_completion.json').is_file())
    if len(runs) != 1:
        raise RuntimeError('Expected one E9 run. Set E9_RESULTS_PATH to its exact folder.')
    return runs[0]


def e9_source_manifest(source):
    if source is None:
        return {'available': False}
    paths = [source/'protocol.json', source/'e9_completion.json']
    for b in FOLD_IDS:
        paths.append(source/'diagnostics'/f'features_fold{b:02d}.json')
        for seed in SEEDS:
            key = f'battery{b:02d}_seed{seed}_DynamicRealGNN'
            paths += [source/'fits'/(key+'.json'),source/'fits'/(key+'_states.csv.gz'),
                      source/'models'/(key+'.pt')]
    files = {str(p.relative_to(source)): sha256_file(p) if p.is_file() else 'MISSING' for p in paths}
    return {'available': True, 'files': files,
            'content_sha256': hashlib.sha256(json.dumps(files,sort_keys=True).encode()).hexdigest()}


def e11_protocol():
    return {'experiment':'E11 canonical-degree adjacency audit','version':E11_VERSION,
        'implementation_sha256':IMPLEMENTATION_SHA256,'source_e9_notebook_sha256':SOURCE_E9_SHA256,
        'dataset_sha256':DATASET_HASHES,'folds':FOLD_IDS,'seeds':SEEDS,'conditions':CONDITIONS,
        'shuffle_ids':SHUFFLE_IDS,'permutation_base_seed':PERMUTATION_BASE_SEED,
        'epochs':EPOCHS,'negative_samples':NEGATIVES_PER_STATE,'learning_rate':LR,
        'weight_decay':WEIGHT_DECAY,'dropout':DROPOUT,'embed_dim':EMBED_DIM,
        'hidden_dim':HIDDEN_DIM,'graph_dim':GNN_DIM,'base_seed':BASE_SEED,
        'cpu_threads':CPU_THREADS,'device':str(DEVICE),'torch_version':torch.__version__,
        'numpy_version':np.__version__,'pandas_version':pd.__version__,'python_version':sys.version.split()[0],
        'deterministic_torch':torch.are_deterministic_algorithms_enabled(),
        'tf32_matmul':torch.backends.cuda.matmul.allow_tf32,
        'degree_rule':'Identical canonical physical node-degree covariate for Real, all shuffles, and NoEdge',
        'activity_rule':'Canonical E6 prefix update; inactive indices fixed; active-only endpoint permutation',
        'permutation_rule':'Two fixed random priority lists per product/realization, restricted to the current active set; unchanged while set unchanged',
        'state_updater':'Original simplified E6 treatment of fixed connections retained',
        'evaluation_batching':EXPECTED_E9_PROTOCOL['evaluation_batching'],
        'independent_unit':'held-out battery product','primary_metric':'hits1',
        'primary_contrasts':['Real - mean of five ActiveShuffle realizations','Real - NoEdge'],
        'holm_family':'Two E11 Hits@1 tests, separate from E10',
        'design_timing':'Core controls specified in the 11 October refinement plan before E10; implementation follows E10; protocol locked before E11 training',
        'save_models':SAVE_MODELS,'reuse_e9_real':REUSE_E9_REAL,'e9_source_manifest':E9_MANIFEST,
        'reuse_rule':'All 30 Real anchors must pass scientific settings, feature/graph identity, software and checkpoint/per-state metric checks; otherwise train all Real fits',
        'sensitivity_rule':'Descriptive Real versus shuffle on states where all five shuffles change the typed adjacency multiset; Real versus NoEdge on states with nonempty canonical edges',
        'scope':'Conditional predictive value within the tested GNN; teacher-forced reference ranking; no causal or planning-feasibility claim'}


def canonical_e11_design():
    return (FOLD_IDS==sorted(chains) and len(FOLD_IDS)==10 and SEEDS==[17,29,43]
        and SHUFFLE_IDS==[1,2,3,4,5] and CONDITIONS==['Real']+[f'ActiveShuffle{i:02d}' for i in SHUFFLE_IDS]+['NoEdge']
        and EPOCHS==30 and NEGATIVES_PER_STATE==8 and LR==2e-3 and WEIGHT_DECAY==1e-4
        and DROPOUT==0.1 and (EMBED_DIM,HIDDEN_DIM,GNN_DIM)==(12,64,32) and BASE_SEED==20261003)


def prepare_run():
    payload = e11_protocol()
    signature = hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()
    run_dir = OUTPUT_ROOT/('run_'+signature[:12])
    run_dir.mkdir(parents=True,exist_ok=True)
    for name in ['fits','models','diagnostics']:
        (run_dir/name).mkdir(exist_ok=True)
    path=run_dir/'protocol.json'
    if path.exists():
        assert json.loads(path.read_text())==payload,'Saved protocol differs.'
    else:
        atomic_json(path,payload)
    return run_dir,signature


def validate_detail(frame,record,b,seed,condition):
    n=len(chains[b])-1
    assert len(frame)==n and record['n_states']==n
    np.testing.assert_array_equal(frame.position,np.arange(n))
    np.testing.assert_array_equal(frame.candidate_size,np.arange(n,0,-1))
    np.testing.assert_allclose(frame.progress,np.arange(n)/n,rtol=0,atol=1e-15)
    assert frame.battery_id.eq(b).all() and frame.seed.eq(seed).all() and frame.condition.eq(condition).all()
    assert np.isfinite(frame[METRICS]).all().all()
    assert ((frame[METRICS]>=0)&(frame[METRICS]<=1)).all().all()
    np.testing.assert_allclose(frame[METRICS].mean(),[record[m] for m in METRICS],rtol=0,atol=1e-12)


def verify_real_anchors(source,run_dir):
    report={'requested':REUSE_E9_REAL,'eligible':False,'checked_fits':0,
            'reason':'No E9 source found; all 30 Real anchors will be trained.'}
    verified={}
    if REUSE_E9_REAL and source is not None:
        try:
            assert canonical_e11_design(),'Anchor reuse requires the complete canonical scientific design.'
            previous=json.loads((source/'protocol.json').read_text())
            for key in ['experiment','feature_version','implementation_sha256','source_e6_sha256',
                        'dataset_sha256','seeds','epochs','negative_samples','learning_rate',
                        'weight_decay','dropout','folds','deterministic_torch','tf32_matmul',
                        'evaluation_batching','state_rule']:
                assert previous[key]==EXPECTED_E9_PROTOCOL[key],'Unexpected E9 field: '+key
            assert previous['dataset_sha256']==DATASET_HASHES
            assert previous['torch_version']==torch.__version__,'Torch version differs; train all Real anchors in the current environment.'
            assert previous['numpy_version']==np.__version__,'NumPy version differs; train all Real anchors in the current environment.'
            assert torch.are_deterministic_algorithms_enabled() and not torch.backends.cuda.matmul.allow_tf32
            completion=json.loads((source/'e9_completion.json').read_text())
            assert completion['finished_fits']==150 and completion['full_canonical_design'] and completion['all_requested_fits_complete']
            old_signature=hashlib.sha256(json.dumps(previous,sort_keys=True).encode()).hexdigest()
            for b in FOLD_IDS:
                train=[x for x in sorted(chains) if x!=b]
                vocab=build_fold_vocab(train)
                old=build_e6_state_pack(b,vocab,'dynamic_real')
                item=build_e11_state_pack(b,vocab,'Real',canonical=old)
                assert_same_graph_inputs(old,item)
                definitions=json.loads((source/'diagnostics'/f'features_fold{b:02d}.json').read_text())
                assert definitions['vocab']==vocab
                for seed in SEEDS:
                    key,path,state_path=fit_paths(source,b,seed,'DynamicRealGNN')
                    record=json.loads(path.read_text());frame=pd.read_csv(state_path)
                    assert record['protocol_signature']==old_signature
                    assert record['battery_id']==b and record['seed']==seed and record['condition']=='DynamicRealGNN'
                    assert record['device']==str(DEVICE),'Device type differs; train all Real anchors.'
                    validate_detail(frame,record,b,seed,'DynamicRealGNN')
                    model_path=source/'models'/(key+'.pt')
                    payload=torch.load(model_path,map_location=DEVICE,weights_only=True)
                    assert payload['vocab']==vocab and payload['condition']=='DynamicRealGNN'
                    model=build_e6_model(vocab)
                    assert count_trainable_parameters(model)==record['parameter_count']==definitions['gnn_parameters']
                    model.load_state_dict(payload['state_dict'],strict=True)
                    metrics,detail=evaluate_e11(model,item)
                    np.testing.assert_allclose(detail[METRICS],frame[METRICS],rtol=0,atol=1e-12)
                    np.testing.assert_allclose([metrics[m] for m in METRICS],[record[m] for m in METRICS],rtol=0,atol=1e-12)
                    verified[(b,seed)]={'record':record,'record_sha256':sha256_file(path),
                        'states_path':state_path,'states_sha256':sha256_file(state_path),
                        'model_path':model_path,'model_sha256':sha256_file(model_path)}
                    report['checked_fits']+=1
                    del model
                print(f'Real-anchor verification: product {b}, {report["checked_fits"]}/30 checked',flush=True)
            assert len(verified)==30
            report.update(eligible=True,reason='All 30 E9 Real anchors passed numerical and procedural checks.')
        except Exception as exc:
            verified={}
            report['reason']=type(exc).__name__+': '+str(exc)
            print('All Real anchors will be trained afresh:',report['reason'],flush=True)
    elif not REUSE_E9_REAL:
        report['reason']='Anchor reuse disabled.'
    report['new_full_budget']=180 if report['eligible'] else 210
    atomic_json(run_dir/'diagnostics'/'real_anchor_verification.json',report)
    return verified,report


def completed_fit(run_dir,b,seed,condition,signature):
    key,path,state_path=fit_paths(run_dir,b,seed,condition)
    if not path.exists() or not state_path.exists():
        return False
    record=json.loads(path.read_text())
    assert record['protocol_signature']==signature
    assert record['battery_id']==b and record['seed']==seed and record['condition']==condition
    if record['origin']=='New E11 training':
        assert record['epochs']==EPOCHS
        curve=record['loss_curve']
        assert len(curve)==EPOCHS and [x['epoch'] for x in curve]==list(range(1,EPOCHS+1))
        assert np.isfinite([x['loss'] for x in curve]).all(),'Invalid saved loss history.'
    assert record['states_sha256']==sha256_file(state_path),'Completed state file changed.'
    validate_detail(pd.read_csv(state_path),record,b,seed,condition)
    if SAVE_MODELS:
        model_path=run_dir/'models'/(key+'.pt')
        assert model_path.is_file() and record['model_sha256']==sha256_file(model_path),'Completed model changed.'
    return True


def commit_fit(run_dir,signature,b,seed,condition,metrics,detail,vocab,extra,model=None,reused=None):
    key,path,state_path=fit_paths(run_dir,b,seed,condition)
    detail=detail.copy()
    detail['battery_id']=b;detail['battery']=str(battery_names[b]);detail['seed']=seed;detail['condition']=condition
    family='ActiveShuffle' if condition.startswith('ActiveShuffle') else condition
    detail['family']=family
    realization=int(condition.removeprefix('ActiveShuffle')) if family=='ActiveShuffle' else 0
    detail['realization']=realization
    record={'protocol_signature':signature,'battery_id':b,'battery':str(battery_names[b]),
        'seed':seed,'condition':condition,'family':family,'realization':realization,
        'n_states':len(detail),**metrics,**extra,
        'origin':'Verified E9 Real anchor; no new training' if reused else 'New E11 training'}
    if reused:
        assert sha256_file(reused['states_path'])==reused['states_sha256'],'E9 states changed after verification.'
        assert sha256_file(reused['model_path'])==reused['model_sha256'],'E9 model changed after verification.'
        record['e9_provenance']={k:reused[k] for k in ['record_sha256','states_sha256','model_sha256']}
        record['e9_provenance']['original_condition']='DynamicRealGNN'
    if SAVE_MODELS:
        model_path=run_dir/'models'/(key+'.pt')
        temporary=model_path.with_name(model_path.stem+'.tmp.pt')
        if reused:
            shutil.copyfile(reused['model_path'],temporary)
        else:
            torch.save({'state_dict':model.state_dict(),'vocab':vocab,'condition':condition,
                        'protocol_signature':signature,'extra':extra},temporary)
        os.replace(temporary,model_path)
        record['model_sha256']=sha256_file(model_path)
    atomic_csv(detail,state_path)
    record['states_sha256']=sha256_file(state_path)
    validate_detail(detail,record,b,seed,condition)
    atomic_json(path,record)


def run_e11():
    run_dir,signature=prepare_run()
    assert run_dir==RUN_DIR and signature==RUN_SIGNATURE
    total=len(FOLD_IDS)*len(SEEDS)*len(CONDITIONS)
    done,new=0,0
    print('E11 fit records:',total,'| canonical new-fit budget:',ANCHOR_REPORT['new_full_budget'],flush=True)
    for b in FOLD_IDS:
        train=[x for x in sorted(chains) if x!=b]
        assert len(train)==9 and b not in train
        vocab=build_fold_vocab(train)
        for condition in CONDITIONS:
            for seed in SEEDS:
                if completed_fit(run_dir,b,seed,condition,signature):
                    done+=1
                    print(f'[{done}/{total}] resumed {b} {seed} {condition}',flush=True)
                    continue
                if condition=='Real' and (b,seed) in VERIFIED_REAL:
                    reused=VERIFIED_REAL[(b,seed)]
                    item=build_e11_state_pack(b,vocab,'Real')
                    old=reused['record'];detail=pd.read_csv(reused['states_path'])
                    detail=detail.merge(item['perturbation_diagnostics'],on='position',validate='one_to_one')
                    commit_fit(run_dir,signature,b,seed,condition,{m:old[m] for m in METRICS},detail,vocab,
                        {'parameter_count':old['parameter_count'],'device':old['device'],
                         'e9_training_runtime_seconds':old['runtime_seconds']},reused=reused)
                    done+=1
                    print(f'[{done}/{total}] imported verified E9 Real {b} {seed}',flush=True)
                    continue
                started=time.time()
                print(f'Starting E11: product {b}, seed {seed}, {condition}',flush=True)
                model,item,model_vocab,losses=train_e11(train,b,seed,condition)
                assert model_vocab==vocab
                metrics,detail=evaluate_e11(model,item)
                count=count_trainable_parameters(model)
                atomic_json(run_dir/'diagnostics'/f'vocab_fold{b:02d}.json',
                            {'vocab':vocab,'parameter_count':count})
                commit_fit(run_dir,signature,b,seed,condition,metrics,detail,vocab,
                    {'parameter_count':count,'epochs':EPOCHS,'loss_curve':losses,
                     'runtime_seconds':time.time()-started,'device':str(DEVICE)},model=model)
                done+=1;new+=1
                print(f'[{done}/{total}] saved {condition}; Hits@1={metrics["hits1"]:.4f}',flush=True)
                del model,item
                gc.collect()
                if torch.cuda.is_available():torch.cuda.empty_cache()
                if MAX_NEW_FITS_THIS_CALL is not None and new>=MAX_NEW_FITS_THIS_CALL:
                    print('Per-call limit reached. Rerun this cell to continue without deleting completed fits.',flush=True)
                    summarize_e11(run_dir)
                    return run_dir
    summarize_e11(run_dir)
    print('All E11 fits are complete. Summary files were written automatically.',flush=True)
    return run_dir


def summarize_e11(run_dir):
    run_dir=Path(run_dir)
    print('Checking completed fits and updating E11 summaries...',flush=True)
    expected=set(itertools.product(FOLD_IDS,SEEDS,CONDITIONS))
    records,frames=[],[]
    for path in sorted((run_dir/'fits').glob('*.json')):
        r=json.loads(path.read_text())
        key=(r['battery_id'],r['seed'],r['condition'])
        assert key in expected,'Unexpected fit in this run directory.'
        assert completed_fit(run_dir,*key,RUN_SIGNATURE)
        _,_,state_path=fit_paths(run_dir,*key)
        frames.append(pd.read_csv(state_path))
        records.append({k:v for k,v in r.items() if k not in ['loss_curve','e9_provenance']})
    keys=[(r['battery_id'],r['seed'],r['condition']) for r in records]
    assert len(keys)==len(set(keys))
    complete=set(keys)==expected
    status={'requested_fits':len(expected),'finished_fits':len(records),
        'all_requested_fits_complete':complete,'full_canonical_design':canonical_e11_design(),
        'verified_e9_real_anchors':sum(r['origin'].startswith('Verified E9') for r in records),
        'new_training_records':sum(r['origin']=='New E11 training' for r in records),
        'primary_inference_enabled':complete and canonical_e11_design(),
        'status':'Complete' if complete else ('Preflight only; no full-budget study fits' if not records else 'Partial study')}
    atomic_json(run_dir/'e11_completion.json',status)
    if not records:
        print('No study fits yet. Smoke scores are diagnostic only.')
        return pd.DataFrame(),pd.DataFrame(),status
    fit=pd.DataFrame(records)
    atomic_csv(fit,run_dir/'e11_fit_results.csv')
    products=[]
    for (b,condition),g in fit.groupby(['battery_id','condition']):
        if set(g.seed)==set(SEEDS):
            products.append({'battery_id':b,'battery':g.battery.iloc[0],'condition':condition,
                'family':g.family.iloc[0],'realization':int(g.realization.iloc[0]),
                **{m:float(g[m].mean()) for m in METRICS}})
    condition_product=pd.DataFrame(products,columns=['battery_id','battery','condition','family','realization',*METRICS])
    atomic_csv(condition_product,run_dir/'e11_condition_product.csv')
    families=[]
    for (b,family),g in condition_product.groupby(['battery_id','family']):
        required={f'ActiveShuffle{i:02d}' for i in SHUFFLE_IDS} if family=='ActiveShuffle' else {family}
        if set(g.condition)==required:
            families.append({'battery_id':b,'battery':g.battery.iloc[0],'family':family,
                             **{m:float(g[m].mean()) for m in METRICS}})
    product=pd.DataFrame(families,columns=['battery_id','battery','family',*METRICS])
    atomic_csv(product,run_dir/'e11_fold_mean.csv')
    aggregate=product.groupby('family',as_index=False).agg(
        hits1=('hits1','mean'),hits3=('hits3','mean'),mrr=('mrr','mean'),n_products=('battery_id','nunique'))
    atomic_csv(aggregate,run_dir/'e11_aggregate.csv')
    contrasts=[]
    for alternative in ['ActiveShuffle','NoEdge']:
        a=product[product.family=='Real'].set_index('battery_id')
        b=product[product.family==alternative].set_index('battery_id')
        shared=sorted(set(a.index)&set(b.index))
        ready=status['primary_inference_enabled'] and set(shared)==set(sorted(chains))
        for metric in METRICS:
            difference=(a.loc[shared,metric]-b.loc[shared,metric]).to_numpy(dtype=float)
            mean=lo=hi=np.nan
            if len(difference):
                mean=float(difference.mean())
                boot=np.random.default_rng(BASE_SEED).choice(difference,(10000,len(difference)),replace=True).mean(axis=1)
                lo,hi=np.quantile(boot,[.025,.975])
            contrasts.append({'contrast':'Real - '+alternative,'metric':metric,'n_products':len(shared),
                'mean_difference':mean,'bootstrap95_low':lo,'bootstrap95_high':hi,
                'product_wins':int((difference>0).sum()),'primary_design_complete':bool(ready),
                'p_exact':exact_sign_flip(difference) if ready else np.nan})
    contrasts=pd.DataFrame(contrasts)
    contrasts['p_holm']=np.nan
    primary=contrasts.metric=='hits1'
    values=contrasts.loc[primary,'p_exact']
    adjusted=holm(values.fillna(1).to_numpy())
    contrasts.loc[primary,'p_holm']=np.where(values.notna(),adjusted,np.nan)
    atomic_csv(contrasts,run_dir/'e11_contrasts.csv')
    details=pd.concat(frames,ignore_index=True)
    # Identical diagnostic flags across seeds for a fixed product/realization.
    flags=['active_nodes','canonical_directed_edges','message_directed_edges','moved_active_nodes',
           'typed_adjacency_changed','trivially_nonperturbable','computational_degree_mismatch_fraction']
    for flag in flags:
        assert details.groupby(['battery_id','condition','position'])[flag].nunique().le(1).all()
    perturbation=details.drop_duplicates(['battery_id','condition','position'])
    diagnostic=perturbation.groupby(['battery_id','condition','family'],as_index=False).agg(
        states=('position','size'),mean_active_nodes=('active_nodes','mean'),
        mean_moved_active_nodes=('moved_active_nodes','mean'),
        typed_adjacency_changed_fraction=('typed_adjacency_changed','mean'),
        trivially_nonperturbable_fraction=('trivially_nonperturbable','mean'),
        mean_degree_mismatch_fraction=('computational_degree_mismatch_fraction','mean'))
    atomic_csv(diagnostic,run_dir/'e11_perturbation_product_summary.csv')
    nonsingleton=details[details.candidate_size>1].groupby(['battery_id','family','seed','realization'],as_index=False)[METRICS].mean()
    nonsingleton=nonsingleton.groupby(['battery_id','family'],as_index=False)[METRICS].mean()
    atomic_csv(nonsingleton,run_dir/'e11_without_singleton_descriptive.csv')
    details['stage']=pd.cut(details.progress,[-1,1/3,2/3,1.001],labels=['early','middle','late'],right=False)
    stages=details.groupby(['battery_id','family','seed','realization','stage'],observed=True,as_index=False)[METRICS].mean()
    stages=stages.groupby(['battery_id','family','stage'],observed=True,as_index=False)[METRICS].mean()
    atomic_csv(stages,run_dir/'e11_stage_product_descriptive.csv')
    # Sensitivities require the complete run; all comparisons use matched states.
    sensitivity=[]
    if status['primary_inference_enabled']:
        seed_means=details.groupby(['battery_id','condition','family','position'],as_index=False)[METRICS].mean()
        for b in FOLD_IDS:
            flags_b=perturbation[perturbation.battery_id==b]
            shuffled=flags_b[flags_b.family=='ActiveShuffle'].pivot(index='position',columns='condition',values='typed_adjacency_changed')
            common_positions=shuffled.index[shuffled.all(axis=1)]
            edge_positions=flags_b[(flags_b.condition=='Real')&(flags_b.canonical_directed_edges>0)].position
            for label,positions,families_to_use in [('all_five_shuffles_change_adjacency',common_positions,['Real','ActiveShuffle']),
                                                   ('nonempty_canonical_edges',edge_positions,['Real','NoEdge'])]:
                for family in families_to_use:
                    rows=seed_means[(seed_means.battery_id==b)&(seed_means.family==family)&seed_means.position.isin(positions)]
                    state_mean=rows.groupby('position')[METRICS].mean()
                    sensitivity.append({'battery_id':b,'sensitivity':label,'family':family,'matched_states':len(state_mean),
                        **{m:float(state_mean[m].mean()) if len(state_mean) else np.nan for m in METRICS}})
    atomic_csv(pd.DataFrame(sensitivity,columns=['battery_id','sensitivity','family','matched_states',*METRICS]),
               run_dir/'e11_matched_state_sensitivity_descriptive.csv')
    display(aggregate)
    display(contrasts[primary])
    print('Completion:',status)
    if not status['primary_inference_enabled']:
        print('Incomplete or diagnostic design: no primary significance values are released.')
    return aggregate,contrasts,status


def export_e11(run_dir):
    # Refresh first: exported archives always contain current completion and summaries.
    aggregate,contrasts,status=summarize_e11(run_dir)
    readme='''E11 canonical-degree adjacency audit
Read e11_completion.json before interpreting the results. The complete design has
210 fit records: Real 30, ActiveShuffle 150 (five realizations), NoEdge 30. Verified E9
Real reuse can reduce new fits to 180. Reduced smoke scores are diagnostics only.
All graph conditions use canonical physical node-degree covariates. Shuffles act only
within the canonical active set; inactive indices, semantic classes, and operation
lookups remain fixed. Two fixed priority lists induce a permutation at each active set.
NoEdge retains degree inputs. The original E6 updater and all fixation records remain.
Permutation can change computational per-node degrees even though supplied covariates
remain canonical. This is a conditional edge/context-alignment test, not a pure causal
decomposition of topology independent of every graph-derived quantity. The no-edge
message branch remains in the nominal architecture but provides no predictive messages.
Average seeds and realizations within product. Two planned primary Hits@1 tests use
one E11 Holm family, separate from E10. Product-level sign exchangeability is assumed;
overlapping LOPO training sets and ten products limit population inference. Bootstrap
intervals, Hits@3/MRR, stage/singleton analyses, and matched-state sensitivities are
descriptive/supporting. Nonsignificance is not equivalence. E11 cannot establish GNN
superiority over the E9 non-graph learners, physical feasibility, or Q1 readiness.
The ZIP contains the protocol, checksums, per-fit/state files, models, and summaries.
Imported Real checkpoints retain their original E9 payload and explicit provenance.
Raw KIT data are excluded. Dataset DOI: 10.35097/emz24pksshndq468.
'''
    (run_dir/'README.txt').write_text(readme,encoding='utf-8')
    if not aggregate.empty:
        fig,ax=plt.subplots(figsize=(7,3))
        plot=aggregate.sort_values('family')
        ax.barh(plot.family,plot.hits1,color='#555555')
        ax.set_xlim(0,1);ax.set_xlabel('Mean product-level Hits@1')
        ax.set_title('E11: '+('complete study' if status['primary_inference_enabled'] else 'partial/diagnostic results'))
        fig.tight_layout();fig.savefig(run_dir/'e11_hits1.png',dpi=180);plt.close(fig)
    path=OUTPUT_ROOT/'e11_canonical_degree_adjacency_results.zip'
    temporary=path.with_suffix('.tmp.zip')
    with zipfile.ZipFile(temporary,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(run_dir.rglob('*')):
            if p.is_file() and '.tmp' not in p.name:z.write(p,str(p.relative_to(run_dir)))
    os.replace(temporary,path)
    print('Result ZIP:',path)
    if IN_COLAB:
        from google.colab import files
        files.download(str(path))
    return path

## Step 6 — Lock the protocol and run required preflight
Verify all states of all ten products for Real, five shuffles, and NoEdge.
Check activity correspondence, degree/class inputs, typed-edge records, graph
degree multisets, unchanged permutations within unchanged active sets, identity
scores, candidate order, and a one-epoch Real training reproduction.
Reduced fits of all seven conditions are diagnostic only. E9 Real verification
occurs after these checks and prints the full new-fit budget before Step 7.

In [ ]:
required=['FOLD_IDS','chains','EXPECTED_E9_PROTOCOL','build_e11_state_pack','resolve_e9_source']
missing=[name for name in required if name not in globals()]
if missing:
    raise RuntimeError('Run Steps 1–5 in order before Step 6. Missing: '+', '.join(missing))
E9_SOURCE=resolve_e9_source() if REUSE_E9_REAL else None
E9_MANIFEST=e9_source_manifest(E9_SOURCE)
RUN_DIR,RUN_SIGNATURE=prepare_run()
print('Protocol locked before E11 training:',RUN_DIR/'protocol.json')
print('E9 source:',E9_SOURCE)
probe_battery=FOLD_IDS[0]
probe_train=[b for b in FOLD_IDS if b!=probe_battery]
probe_vocab=build_fold_vocab(probe_train)
all_diagnostics=[]
for b in FOLD_IDS:
    canonical=build_e6_state_pack(b,probe_vocab,'dynamic_real')
    real=build_e11_state_pack(b,probe_vocab,'Real',canonical=canonical)
    assert_same_graph_inputs(canonical,real)
    identity=build_e11_state_pack(b,probe_vocab,'ActiveShuffle01',identity=True,canonical=canonical)
    assert_same_graph_inputs(real,identity)
    for condition in CONDITIONS:
        item=real if condition=='Real' else build_e11_state_pack(b,probe_vocab,condition,canonical=canonical)
        for key in ['node_degree','node_part_class']:
            np.testing.assert_array_equal(item['state_graph_pack'][key],real['state_graph_pack'][key])
        np.testing.assert_array_equal(item['canonical_active'],real['canonical_active'])
        for key in ['op_numeric','op_categorical','op_graph_nodes']:
            np.testing.assert_array_equal(item['base_pack'][key],real['base_pack'][key])
        if condition=='NoEdge':
            assert item['state_graph_pack']['edge_index'].shape==(2,0)
            assert not item['state_graph_pack']['edge_type'].size
            assert np.count_nonzero(item['state_graph_pack']['node_degree'])==np.count_nonzero(real['state_graph_pack']['node_degree'])
        elif condition.startswith('ActiveShuffle'):
            assert item['state_graph_pack']['edge_index'].shape==real['state_graph_pack']['edge_index'].shape
            np.testing.assert_array_equal(item['state_graph_pack']['edge_type'],real['state_graph_pack']['edge_type'])
            for t in range(1,item['n_states']):
                if np.array_equal(item['canonical_active'][t],item['canonical_active'][t-1]):
                    np.testing.assert_array_equal(item['active_permutations'][t],item['active_permutations'][t-1])
        diag=item['perturbation_diagnostics'].copy()
        diag['battery_id']=b;diag['condition']=condition
        all_diagnostics.append(diag)
    print('Graph/input invariants passed for product',b,flush=True)
CONTROL_DIAGNOSTICS=pd.concat(all_diagnostics,ignore_index=True)
atomic_csv(CONTROL_DIAGNOSTICS,RUN_DIR/'diagnostics'/'preflight_all_graph_controls.csv')
smoke_rows=[]
for i,condition in enumerate(CONDITIONS):
    print(f'Smoke fit {i+1}/{len(CONDITIONS)}: {condition}, one epoch only',flush=True)
    model,item,vocab,losses=train_e11(probe_train,probe_battery,SEEDS[0],condition,epochs=1)
    metrics,detail=evaluate_e11(model,item)
    reordered,_=evaluate_e11(model,item,candidate_seed=89)
    np.testing.assert_array_equal([metrics[m] for m in METRICS],[reordered[m] for m in METRICS])
    assert len(losses)==1 and np.isfinite(losses[0]['loss'])
    if condition=='Real':
        identity=build_e11_state_pack(probe_battery,vocab,'ActiveShuffle01',identity=True)
        compare_raw_scores(model,item,identity)
        saved_epochs=EPOCHS
        try:
            EPOCHS=1
            legacy,legacy_item,legacy_vocab=train_one_e6_condition(probe_train,probe_battery,SEEDS[0],'dynamic_real')
        finally:
            EPOCHS=saved_epochs
        assert legacy_vocab==vocab
        for name,value in model.state_dict().items():
            assert torch.equal(value,legacy.state_dict()[name]),'Real training changed: '+name
        legacy_metrics,_=evaluate_scalar_or_gnn(legacy,legacy_item,'gnn')
        np.testing.assert_array_equal([metrics[m] for m in METRICS],[legacy_metrics[m] for m in METRICS])
        del legacy
    smoke_rows.append({'condition':condition,'diagnostic_only':True,
                       'parameter_count':count_trainable_parameters(model),**metrics})
    del model,item
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
assert len({r['parameter_count'] for r in smoke_rows})==1
atomic_csv(pd.DataFrame(smoke_rows),RUN_DIR/'diagnostics'/'smoke_scores_not_study_results.csv')
VERIFIED_REAL,ANCHOR_REPORT=verify_real_anchors(E9_SOURCE,RUN_DIR)
display(pd.DataFrame(smoke_rows))
print('Real-anchor verification:',ANCHOR_REPORT)
PREFLIGHT_PASSED=True
print('Preflight passed. Step 7 performs the actual E11 study.')

## Step 7 — Actual full E11 experiment with resume
The default runs all fits. Progress appears at each start and completion.
Completed files are saved on Drive after every fit. Re-run this cell to continue
after interruption or a per-call fit limit. Summary files are produced automatically
when the run finishes or reaches the configured per-call limit.

In [ ]:
if not globals().get('PREFLIGHT_PASSED',False):
    raise RuntimeError('Complete Step 6 before running E11.')
if RUN_FULL_EXPERIMENT:
    RUN_DIR=run_e11()
else:
    summarize_e11(RUN_DIR)
    print('Explicit preflight-only mode: RUN_FULL_EXPERIMENT=False.')

## Step 8 — Refresh summaries and export the result ZIP
This cell refreshes completion and analysis files before exporting, even if an
earlier summary cell was skipped. A partial run remains labelled incomplete.
Keep the exported ZIP; it contains raw states, models, hashes, and diagnostics.

In [ ]:
if not globals().get('PREFLIGHT_PASSED',False):
    raise RuntimeError('Run Step 6 before exporting.')
ZIP_PATH=export_e11(RUN_DIR)

## Step 9 — Planned interpretation and next manuscript revision
The primary comparisons are Real minus the five-realization ActiveShuffle mean
and Real minus NoEdge. Average seeds and shuffle realizations within product,
then compare ten product differences. Both primary Hits@1 tests share one E11
Holm family, separate from E10. Bootstrap intervals, Hits@3/MRR, stage, singleton,
and matched-state sensitivities are supporting/descriptive. Overlapping LOPO
training sets and the small product collection limit population inference.

A positive Real-minus-shuffle result supports edge correspondence with canonical
physical node context. The shuffle can change computational per-node degrees
while keeping supplied degree covariates fixed; this is not a pure causal
decomposition of edge placement independent of all structural quantities.
A positive Real-minus-NoEdge result supports message passing within this GNN.
A negative result should be reported as such. Unsupported differences are not
proof of equivalence. All identity/unchanged states stay in the primary evaluation.

E9 and E10 already show that useful physical information can be expressed through
evolving local summaries in non-graph learners. E11 addresses the conditional
value of adjacency inside the tested GNN; it does not establish that this GNN
outperforms those learners. Revise the article around the complete evidence and
compare the joint audit with the closest prior studies before claiming novelty.

In [ ]:
status_path=RUN_DIR/'e11_completion.json'
if status_path.exists():
    status=json.loads(status_path.read_text())
    print('Study status:',status)
    if status.get('primary_inference_enabled'):
        print('The full canonical E11 design is complete. Share ZIP_PATH for an independent audit.')
    else:
        print('Continue Step 7 to complete the study; current diagnostics/partial outputs do not support primary inference.')
else:
    print('Run Steps 7–8 to create completion and export files.')